In [22]:
import pandas as pd
import numpy as np

In [23]:
# Cargar las bases de datos
df_res = pd.read_csv('Base_res_coop.csv')
df_dem = pd.read_csv('Base_dem_coop.csv')
df_info_bloques = pd.read_csv('Inform4ción blo#ues.csv')

In [24]:
# 2. Pre-procesamiento de Info Bloques para el merge
# Renombramos 'Nombre' a 'Bloque' para que coincida con df_res_dict
df_info_bloques = df_info_bloques.rename(columns={'Nombre': 'Bloque'})
# Seleccionamos variables de interés (Gap)
# Usamos 'Diferencia desigualdad final entre opciones' como el costo de revertir
cols_interes = ['Bloque', 'Diferencia desigualdad final entre opciones']
df_info_bloques_sel = df_info_bloques[cols_interes]


In [4]:
# --- PASO 1: Filtro de Atención ---
# Identificar sujetos que fallaron el test de atención
# Condición: Dilema es 'Atencion' y Respuesta NO es 'Opción 3'
sujetos_fallaron = df_res[
    (df_res['Dilema'] == 'Atencion') & 
    (df_res['Respuesta'] != 'Opción 3')
]['ID_Sujeto'].unique()

print(f"Cantidad de sujetos que fallaron atención: {len(sujetos_fallaron)}")
# Opcional: ver los IDs
# print(sujetos_fallaron)

# --- PASO 2: Filtro y Descriptiva de Género ---
# Mostrar descriptiva inicial de género
print("\n--- Descriptiva de Género (Antes del filtro) ---")
print(df_dem['Genero'].value_counts())

# Eliminar personas que no sean 'Hombre' o 'Mujer'
df_dem_filtrado = df_dem[df_dem['Genero'].isin(['Hombre', 'Mujer'])]

df_dem_filtrado = df_dem_filtrado.drop(columns=['Origen_Form'])
# --- PASO 3: Unir las Bases ---
# Primero, filtramos la base de respuestas para quitar a los que fallaron atención
df_res_limpio = df_res[~df_res['ID_Sujeto'].isin(sujetos_fallaron)]

# Unimos usando 'inner' para conservar solo sujetos que sobreviven a AMBOS filtros
# (que pasaron atención Y que tienen género binario)
df_final = pd.merge(df_res_limpio, df_dem_filtrado, on='ID_Sujeto', how='inner')

print(f"\nDimensiones de la base final unida: {df_final.shape}")



Cantidad de sujetos que fallaron atención: 5

--- Descriptiva de Género (Antes del filtro) ---
Mujer         24
Hombre        21
No binario     1
Name: Genero, dtype: int64

Dimensiones de la base final unida: (800, 30)


In [25]:
# --- PASO 4: Crear Variable Dependiente Binaria ---
# 1 si eligió Cooperar, 0 si eligió No cooperar
# Usamos map para asignar los valores. 
# Nota: Las filas de 'Atencion' quedarán como NaN (vacías) con este método, 
# lo cual es correcto si solo vas a analizar los dilemas reales.
df_dem = df_dem.drop(columns=['Origen_Form'])
df_final = pd.merge(df_res, df_dem, on='ID_Sujeto', how='inner')
df_final['Cooperar'] = df_final['Respuesta'].map({
    'Cooperar': 1, 
    'No cooperar': 0
})

# Verificar la creación de la variable
print("\n--- Chequeo de la nueva variable dependiente ---")
print(df_final[['Respuesta', 'Cooperar']].head(10))


--- Chequeo de la nueva variable dependiente ---
     Respuesta  Cooperar
0     Cooperar       1.0
1  No cooperar       0.0
2  No cooperar       0.0
3     Cooperar       1.0
4  No cooperar       0.0
5     Cooperar       1.0
6     Opción 2       NaN
7  No cooperar       0.0
8     Cooperar       1.0
9     Cooperar       1.0


In [26]:
mapeo_expectativas = {
    "No espero ningún efecto": 0,
    "La Opción 1 incrementa la probabilidad de cooperación": 1,
    "La opción 1 incrementa la posibilidad de cooperación" : 1,
    "La Opción 2 incrementa la probabilidad de cooperación": -1,
    "La opción 2 incrementa la posibilidad de cooperación": -1
}

cols_expectativas = ['expectativa_sin', 'expectativa_grande', 'expectativa_pequeña']

# Aplicamos el reemplazo
for col in cols_expectativas:
    if col in df_final.columns:
        df_final[col] = df_final[col].replace(mapeo_expectativas)

In [27]:
# 3. Calcular Scores Psicométricos (SDO y NDC)
# Definir grupos de columnas
sdo_cols = [c for c in df_final.columns if 'sdo_' in c]
ndc_cols = [c for c in df_final.columns if 'ndc_' in c]

# Ítems a invertir
items_inv_ndc = ['ndc_3', 'ndc_4']
items_inv_sdo = ['sdo_2', 'sdo_4', 'sdo_6', 'sdo_8', 'sdo_10']

# Aplicar inversión para NDC (Escala 1-7 -> n+1 = 8)
for col in items_inv_ndc:
    df_final[col] = 8 - df_final[col]

# Aplicar inversión para SDO (Escala 1-5 -> n+1 = 6)
for col in items_inv_sdo:
    df_final[col] = 6 - df_final[col]

# Calcular Scores Psicométricos (Promedios)
df_final['SDO_Score'] = df_final[sdo_cols].mean(axis=1)
df_final['NDC_Score'] = df_final[ndc_cols].mean(axis=1)

items_indiv = ['ndc_1','ndc_2','ndc_3',
 'ndc_4',
 'ndc_5',
 'ndc_6',
 'sdo_1',
 'sdo_2',
 'sdo_3',
 'sdo_4',
 'sdo_5',
 'sdo_6',
 'sdo_7',
 'sdo_8',
 'sdo_9',
 'sdo_10']

df_final= df_final.drop (columns=items_indiv)


In [28]:
df_final = df_final.merge(df_info_bloques_sel, on='Bloque', how='left')
df_final.rename(columns={'Diferencia desigualdad final entre opciones': 'Gap_Size'}, inplace=True)
df_final.rename(columns={'Categoria': 'orden'}, inplace=True)

In [29]:
df_final.columns.tolist()

['ID_Sujeto',
 'Origen_Form',
 'Identidad',
 'Bloque',
 'Dilema',
 'Dilema_Opcion',
 'orden',
 'Respuesta',
 'expectativa_sin',
 'expectativa_grande',
 'expectativa_pequeña',
 'Genero',
 'politica',
 'nivel_se',
 'Cooperar',
 'SDO_Score',
 'NDC_Score',
 'Gap_Size']

In [30]:
df_final.to_csv('Base_coop_con_todo.csv', index=False)
print("💾 Guardado como 'Base_coop.csv'")

💾 Guardado como 'Base_coop.csv'
